# Ollama — Abliterated Coding Model
Run cells top to bottom.

**Runtime → Change runtime type → GPU**
- T4 (16GB) → runs 14B model
- 2x T4 (32GB) → runs 32B model (better)
- A100 (40GB) → runs 32B model

In [ ]:
# Cell 1 — Check VRAM and pick model
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                        capture_output=True, text=True)
vram_list = [int(x.strip()) for x in result.stdout.strip().split('\n') if x.strip()]
total_vram = sum(vram_list)
print(f'GPUs: {len(vram_list)} × {vram_list[0]}MB = {total_vram}MB total VRAM')

if total_vram >= 28000:
    MODEL = 'huihui_ai/qwen2.5-coder-32b-abliterated'
    print('✓ Using 32B model (best quality)')
else:
    MODEL = 'huihui_ai/qwen2.5-coder-14b-abliterated'
    print('✓ Using 14B model (fits 16GB VRAM)')
print(f'Model: {MODEL}')

In [ ]:
# Cell 2 — Install + start Ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print('Ollama running ✓')

In [ ]:
# Cell 3 — Pull the model (10-20 min depending on size)
!ollama pull {MODEL}

In [ ]:
# Cell 4 — Expose via ngrok so you can call it from your IDE / VPS
!pip install pyngrok -q
from pyngrok import ngrok
# Optional: add free ngrok token from ngrok.com to avoid connection limits
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')
public_url = ngrok.connect(11434)
print(f'\n✓ Ollama API: {public_url}')
print(f'  Model: {MODEL}')
print('  Paste this URL in Continue / Cursor / VS Code as your Ollama base URL')

In [ ]:
# Cell 5 — Test it
import requests
resp = requests.post('http://localhost:11434/api/generate', json={
    'model': MODEL,
    'prompt': 'Write a Python port scanner.',
    'stream': False
})
print(resp.json()['response'])

In [ ]:
# Cell 6 — Keep session alive (prevents Colab from disconnecting)
import time
print('Keeping session alive... interrupt kernel to stop.')
while True:
    time.sleep(60)
    print('.', end='', flush=True)